In [5]:
import pandas as pd
from pathlib import Path

files = sorted(Path("/Users/kellyg/eurogate-twin-1/eurogate_daily").glob("*.csv"))

for f in files[:5]:
    df = pd.read_csv(f, nrows=5, encoding="latin1")
    print(f.name)
    print(df.columns.tolist())
    print(df.head())
    print("-"*100)

Containerdaten-2024-10-27_01.55.36.csv
['containerId', 'customsRefId', 'lineCode', 'iedCode', 'sizetypeIsoCode', 'typeCode', 'reefer', 'teu', 'mgw', 'gross', 'csc', 'originOfTransportCode', 'arrivalTime', 'arrivalPolCode', 'arrivalType', 'arrivalLocationX', 'arrivalLocationY', 'arrivalLocationZ', 'arrivalVoyageEta', 'arrivalVesselName', 'arrivalServiceName', 'arrivalServiceCode', 'departureTime', 'departureType', 'destinationCode', 'released', 'forwarderName', 'numberOfArrivalContainersOnVessel', 'numberOfRemainOnBoardContainersOnVessel', 'numberOfDepartureContainersOnVessel']
   containerId           customsRefId  lineCode iedCode sizetypeIsoCode  \
0  TCNU2197105  ATB151468130920244851        77  IMPORT            45G1   
1  TGBU8605210  ATB150644481020244851        32  IMPORT            45G1   
2  GCXU6111154  ATB150553101020244851        77  IMPORT            45G1   
3  MEDU8987629  ATB150630421020244851        68  IMPORT            45G1   
4  CMAU5670653  ATB150564421020244851    

In [10]:
# # !pwd
# print(df.shape)
# print(df.columns.tolist())

import pandas as pd

df1 = pd.read_csv(files[0], encoding="latin1")
df2 = pd.read_csv(files[1], encoding="latin1")

overlap = set(df1["containerId"]).intersection(
    set(df2["containerId"])
)
# print(df1)
print(len(overlap))

17607


In [11]:
import pandas as pd
from pathlib import Path

folder = Path("/Users/kellyg/eurogate-twin-1/eurogate_daily")
files = sorted(folder.glob("*.csv"))

daily_dfs = []

for f in files:
    df = pd.read_csv(f, encoding="latin1")
    
    # Extract snapshot date from filename
    # Example: Containerdaten-2024-10-27_01.55.36.csv
    snapshot_str = f.name.replace("Containerdaten-", "").split("_")[0]
    df["snapshot_date"] = pd.to_datetime(snapshot_str)
    df["source_file"] = f.name
    
    daily_dfs.append(df)

all_snapshots = pd.concat(daily_dfs, ignore_index=True)

print(all_snapshots.shape)
print(all_snapshots["snapshot_date"].min(), all_snapshots["snapshot_date"].max())
print(all_snapshots["containerId"].nunique())

# Parse dates
for col in ["arrivalTime", "departureTime", "arrivalVoyageEta"]:
    all_snapshots[col] = pd.to_datetime(all_snapshots[col], errors="coerce")

# Sort by container and snapshot
all_snapshots = all_snapshots.sort_values(["containerId", "snapshot_date"])

container_history = (
    all_snapshots
    .groupby("containerId")
    .agg(
        first_seen=("snapshot_date", "min"),
        last_seen=("snapshot_date", "max"),
        n_snapshots=("snapshot_date", "nunique"),

        arrivalTime=("arrivalTime", "min"),
        departureTime=("departureTime", "max"),
        arrivalVoyageEta=("arrivalVoyageEta", "min"),

        sizetypeIsoCode=("sizetypeIsoCode", "first"),
        typeCode=("typeCode", "first"),
        reefer=("reefer", "first"),
        teu=("teu", "first"),
        gross=("gross", "first"),
        lineCode=("lineCode", "first"),
        arrivalType=("arrivalType", "first"),
        departureType=("departureType", "first"),
        originOfTransportCode=("originOfTransportCode", "first"),
        arrivalPolCode=("arrivalPolCode", "first"),
        destinationCode=("destinationCode", "first"),
        released=("released", "last"),

        n_unique_locations_x=("arrivalLocationX", "nunique"),
        n_unique_locations_y=("arrivalLocationY", "nunique"),
        n_unique_locations_z=("arrivalLocationZ", "nunique"),
    )
    .reset_index()
)

container_history["observed_days"] = (
    container_history["last_seen"] - container_history["first_seen"]
).dt.days + 1

container_history["dwell_hours"] = (
    container_history["departureTime"] - container_history["arrivalTime"]
).dt.total_seconds() / 3600

container_history["vessel_eta_delay_hours"] = (
    container_history["arrivalTime"] - container_history["arrivalVoyageEta"]
).dt.total_seconds() / 3600

container_history.head()

loc_cols = ["arrivalLocationX", "arrivalLocationY", "arrivalLocationZ"]

all_snapshots["location_key"] = (
    all_snapshots[loc_cols]
    .astype(str)
    .agg("_".join, axis=1)
)

movement_counts = (
    all_snapshots
    .sort_values(["containerId", "snapshot_date"])
    .assign(prev_location=lambda x: x.groupby("containerId")["location_key"].shift())
)

movement_counts["location_changed"] = (
    movement_counts["location_key"] != movement_counts["prev_location"]
) & movement_counts["prev_location"].notna()

rehandle_summary = (
    movement_counts
    .groupby("containerId")
    .agg(
        n_observed_location_changes=("location_changed", "sum"),
        n_unique_locations=("location_key", "nunique")
    )
    .reset_index()
)

container_history = container_history.merge(rehandle_summary, on="containerId", how="left")

container_history.to_csv("eurogate_container_history.csv", index=False)
all_snapshots.to_csv("eurogate_all_snapshots.csv", index=False)

(1138839, 32)
2024-10-27 00:00:00 2025-05-06 00:00:00
146674


In [6]:
df = pd.read_csv("mega_training_data.csv")

# there could be leakage from exact arrival time to dwell hours
# so just change it a little to make it more training friendly
datetime_cols = [
    "arrival_time",
    "arrival_voyage_eta",
    "first_seen"
]

for col in datetime_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce", utc=True)


df["arrival_hour"] = df["arrival_time"].dt.hour

df["arrival_dayofweek"] = df["arrival_time"].dt.dayofweek
# Monday=0 ... Sunday=6
df["arrival_month"] = df["arrival_time"].dt.month
df["arrival_quarter"] = df["arrival_time"].dt.quarter
df["arrival_is_weekend"] = (
    df["arrival_dayofweek"] >= 5
).astype(int)

# ETA accuracy feature
df["eta_error_hours"] = (
    df["arrival_time"] -
    df["arrival_voyage_eta"]
).dt.total_seconds() / 3600

# Seasonal features
df["arrival_dayofyear"] = (
    df["arrival_time"]
      .dt.dayofyear
)

# cyclical encoding for month
df["month_sin"] = np.sin(
    2 * np.pi * df["arrival_month"] / 12
)
df["month_cos"] = np.cos(
    2 * np.pi * df["arrival_month"] / 12
)
# cyclical encoding for hour

df["hour_sin"] = np.sin(
    2 * np.pi * df["arrival_hour"] / 24
)

df["hour_cos"] = np.cos(
    2 * np.pi * df["arrival_hour"] / 24
)

# Drop original timestamps
cols_to_drop = [
    "arrival_time",
    "arrival_voyage_eta",
    "first_seen"
]

cols_to_drop = [
    c for c in cols_to_drop
    if c in df.columns
]

df = df.drop(columns=cols_to_drop)

df.to_csv(
    "mega_container_dwell_dataset_engineered.csv",
    index=False
)

print(df.shape)

print(
    df[
        [
            "arrival_hour",
            "arrival_dayofweek",
            "arrival_month",
            "arrival_is_weekend",
            "eta_error_hours"
        ]
    ].head()
)

/var/folders/rv/q52q8kmj0s558xpyhlshtzw00000gn/T/ipykernel_32191/3659909785.py:1: DtypeWarning: Columns (7,8,38,39,41,42,44,45,46,47,48) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("mega_training_data.csv")


(221670, 64)
   arrival_hour  arrival_dayofweek  arrival_month  arrival_is_weekend  \
0           0.0                0.0            6.0                   0   
1           0.0                6.0            9.0                   1   
2           0.0                1.0            9.0                   0   
3           0.0                1.0            9.0                   0   
4           0.0                1.0            9.0                   0   

   eta_error_hours  
0              NaN  
1              NaN  
2              NaN  
3              NaN  
4              NaN  


In [17]:
# drop column from mega_training_data.csv: container_id
df = pd.read_csv("mega_training_data.csv")
df = df.drop(columns=["container_id"])
df = df.drop(columns=["departure_time"])
df = df.drop(columns=["n_snapshots"])
df = df.drop(columns=["sizetypeIsoCode"])
df = df.drop(columns=["last_seen"])
df = df.drop(columns=["typeCode"])
df = df.drop(columns=["n_unique_locations_x"])
df = df.drop(columns=["n_unique_locations_y"])
df = df.drop(columns=["n_unique_locations_z"])
df = df.drop(columns=["n_observed_location_changes"])
df = df.drop(columns=["n_unique_locations"])
df = df.drop(columns=["arrival_month"])
df = df.drop(columns=["arrival_quarter"])

# save mega_training_data.csv
df.to_csv("mega_training_data.csv", index=False)

/var/folders/rv/q52q8kmj0s558xpyhlshtzw00000gn/T/ipykernel_32191/1234591468.py:2: DtypeWarning: Columns (6,36,38,39,41,42,43,44,45) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("mega_training_data.csv")


In [22]:
# count how many columns are in mega_training_data.csv
print((df.columns))

Index(['port', 'dwell_hours', 'container_size', 'gross_weight', 'reefer',
       'temperature_2m', 'relative_humidity_2m', 'precipitation',
       'weather_code', 'cloud_cover', 'wind_speed_10m', 'wind_gusts_10m',
       'n_events', 'n_unique_activities', 'n_unique_roles',
       'hours_atb_to_discharge', 'flag_quarantine', 'flag_customs',
       'flag_damage', 'yard_slot', 'yard_block_enc', 'CTR_TYPE_DRY',
       'CTR_TYPE_FLT', 'CTR_TYPE_O/T', 'CTR_TYPE_OVD', 'CTR_TYPE_TNK',
       'JOB_DEL_DOCTYPE_BC23', 'JOB_DEL_DOCTYPE_BCF26',
       'JOB_DEL_DOCTYPE_LAIN2', 'JOB_DEL_DOCTYPE_LELANG',
       'JOB_DEL_DOCTYPE_NNMITA', 'JOB_DEL_DOCTYPE_PLP', 'JOB_DEL_DOCTYPE_SPPB',
       'JOB_DEL_DOCTYPE_TMBLU', 'line_code', 'arrival_type', 'departure_type',
       'origin_transport_code', 'arrival_pol_code', 'released',
       'vessel_eta_delay_hours', 'arrival_hour', 'arrival_dayofweek',
       'arrival_is_weekend', 'eta_error_hours', 'arrival_dayofyear',
       'month_sin', 'month_cos', 'hour_sin

In [20]:
# drop observed days from mega_training_data.csv
df = df.drop(columns=["observed_days"])
# save mega_training_data.csv
df.to_csv("mega_training_data.csv", index=False)
# count how many columns are in mega_training_data.csv
print(len(df.columns))


50
